# Northern accuracy assessment: fetch the 570 reference curves

Fetches each sample point's Sentinel-2 history (Sep 2025 – Aug 2026) from Planetary Computer with the southern assessment's `fetch_curves.py` (clipped NDVI formula). Reads only `data/points.csv` (id, lat, lon), never the map key.

**Runtime ▸ Run all.** Google asks for Drive access: tick every box. About 1.5 hours, no login to Copernicus, no credits.

The results are saved in Drive as they come in (`MyDrive/iraq-farms-health/accuracy_north/curves.json`). If Colab disconnects, just **Run all** again: finished points are skipped.

When it's done, the last cell downloads `curves.json`. Send that file to Claude.

In [ ]:
# 1. Drive (so a disconnect loses nothing)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/iraq-farms-health/accuracy_north'
os.makedirs(OUT_DIR, exist_ok=True)
OUT = OUT_DIR + '/curves.json'
print('results go to', OUT)

In [ ]:
# 2. Only the two assessment folders from the repo (a few MB)
!rm -rf /content/ifh && git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rawazrauf/iraq-farms-health.git /content/ifh
!cd /content/ifh && git sparse-checkout set validation/accuracy_south_2026 validation/accuracy_north_2026 && git log -1 --format='repo at %h  %ad  %s' --date=short
%cd /content/ifh/validation/accuracy_north_2026
!wc -l data/points.csv && grep -n 'def ndvi_cor' ../accuracy_south_2026/fetch_curves.py

In [ ]:
# 3. Fetch (resumes if run again)
!python ../accuracy_south_2026/fetch_curves.py --points data/points.csv --out "$OUT" --workers 3

In [ ]:
# 4. Check and download
import json
from collections import Counter
d = json.load(open(OUT))
err = [k for k, r in d.items() if 'err' in r]
n = Counter(sum('cor' in m for m in r['months'].values()) for r in d.values() if 'months' in r)
print(f'{len(d)} of 570 points; errors: {len(err)} {err[:5]}')
print('clear months per point:', dict(sorted(n.items(), reverse=True)))
if len(d) == 570 and not err:
    from google.colab import files
    files.download(OUT)
else:
    print('Not complete yet: run cell 3 again (finished points are skipped), then this cell.')